In [1]:
# Cell 1
import os, sys, json
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    sys.path.insert(0, project_root)

from config import get_driver
from src.ci.indication_service import (
    create_indication, link_program_to_indication, list_indications)
from src.ci.product_asset_service import (
    create_product_asset, list_product_assets)
from src.decision_support.decision_option_service import (
    create_decision_option, list_options_for_brief)
from src.engineering_intelligence.payload_service import (
    create_delivery_payload, list_payloads)
from src.engineering_intelligence.modification_service import (
    create_engineering_modification, list_modifications)

driver = get_driver()
print("✅ 环境就绪")

✅ 环境就绪


In [2]:
# Cell 2: ClinicalIndication
ind1 = create_indication(
    driver,
    name="Diabetic foot osteomyelitis",
    infection_site="Diabetic foot",
    pathogen_scope="Staphylococcus aureus / Klebsiella pneumoniae",
    disease_stage="chronic",
    standard_term="DFO",
    actor_id="demo",
)
ind2 = create_indication(
    driver,
    name="Cystic fibrosis lung infection",
    infection_site="Lung",
    pathogen_scope="Pseudomonas aeruginosa",
    disease_stage="chronic",
    standard_term="CF-PA",
    actor_id="demo",
)
print(f"✅ indications: {ind1}, {ind2}")

# 关联一个项目
with driver.session() as s:
    prog = s.run("""
        MATCH (d:DevelopmentProgram)
        RETURN d.program_id AS pid, d.canonical_name AS name
        ORDER BY d.canonical_name LIMIT 1
    """).single()
if prog:
    link_program_to_indication(driver, prog["pid"], ind1, actor_id="demo")
    print(f"✅ {prog['name']} -> DFO")

print("\n=== 所有 ClinicalIndication ===")
for it in list_indications(driver):
    print(f"  {it['indication_id']}  {it['name']}  程序数={it['program_count']}")

✅ indications: CI:IND:8FC40A70, CI:IND:03236DFE
✅ Athinia -> DFO

=== 所有 ClinicalIndication ===
  CI:IND:03236DFE  Cystic fibrosis lung infection  程序数=0
  CI:IND:8FC40A70  Diabetic foot osteomyelitis  程序数=1
  CI:IND:AEDBADFC  验收适应症-Demo  程序数=1


In [3]:
# Cell 3: ProductAsset
with driver.session() as s:
    org = s.run("""
        MATCH (o:Organization)-[:DEVELOPS]->(:DevelopmentProgram)
        RETURN o.organization_id AS oid, o.canonical_name AS name
        ORDER BY o.canonical_name LIMIT 1
    """).single()

asset_id = create_product_asset(
    driver,
    name=f"{org['name']} PhageBank Product",
    organization_id=org["oid"],
    asset_type="drug",
    status="clinical",
    description="PhageBank productized asset for MDR infections",
    actor_id="demo",
)
print(f"✅ ProductAsset: {asset_id}")

print("\n=== 所有 ProductAsset ===")
for a in list_product_assets(driver):
    print(f"  {a['asset_id']}  {a['name']}  [{a['asset_type']}/{a['status']}]  归属={a['organization']}")

✅ ProductAsset: CI:ASSET:3F59B275

=== 所有 ProductAsset ===
  CI:ASSET:3F59B275  Adaptive Phage Therapeutics PhageBank Product  [drug/clinical]  归属=Adaptive Phage Therapeutics
  CI:ASSET:E512CE33  验收资产-Demo  [drug/clinical]  归属=Adaptive Phage Therapeutics


In [4]:
# Cell 4: DecisionOption
with driver.session() as s:
    brief = s.run("""
        MATCH (b:IntelligenceProduct)
        RETURN b.brief_id AS bid, b.title AS title
        ORDER BY b.created_at DESC LIMIT 1
    """).single()

print(f"目标 Brief: {brief['title']} ({brief['bid']})")

opt1 = create_decision_option(
    driver,
    brief_id=brief["bid"],
    option_type="monitor",
    option_summary="持续监控该对手的临床与专利进展",
    pros=["成本低", "保持信息更新"],
    cons=["错过窗口期风险"],
    risk_level="low",
    actor_id="demo",
)
opt2 = create_decision_option(
    driver,
    brief_id=brief["bid"],
    option_type="evaluate",
    option_summary="启动内部可行性评估（含 FTO 初筛）",
    pros=["抢占先机", "技术路线验证"],
    cons=["资源投入", "结果不确定"],
    risk_level="medium",
    actor_id="demo",
)
print(f"✅ options: {opt1}, {opt2}")

print("\n=== Brief 的所有 DecisionOption ===")
for o in list_options_for_brief(driver, brief["bid"]):
    print(f"  {o['option_id']}  [{o['option_type']}/{o['risk_level']}]  {o['option_summary']}")

目标 Brief: 技术情报简报 - biofilm_disruption (CI:BRIEF:FA60BDE1)
✅ options: CI:OPT:26E31822, CI:OPT:BF2B92F9

=== Brief 的所有 DecisionOption ===
  CI:OPT:26E31822  [monitor/low]  持续监控该对手的临床与专利进展
  CI:OPT:BF2B92F9  [evaluate/medium]  启动内部可行性评估（含 FTO 初筛）


In [5]:
# Cell 5: DeliveryPayload
with driver.session() as s:
    cid = s.run("""
        MATCH (ec:EngineeredPhageConstruct)
        RETURN ec.construct_id AS cid, ec.public_name AS name
        ORDER BY ec.public_name LIMIT 1
    """).single()

print(f"目标构建体: {cid['name']} ({cid['cid']})")

pl1 = create_delivery_payload(
    driver,
    payload_type="biofilm_enzyme",
    payload_description="Displays biofilm-degrading enzyme on phage capsid",
    construct_id=cid["cid"],
    actor_id="demo",
)
pl2 = create_delivery_payload(
    driver,
    payload_type="crispr_cas",
    payload_description="Encodes CRISPR-Cas3 to target antibiotic resistance genes",
    construct_id=cid["cid"],
    actor_id="demo",
)
print(f"✅ payloads: {pl1}, {pl2}")

print("\n=== 所有 DeliveryPayload ===")
for p in list_payloads(driver):
    print(f"  {p['payload_id']}  [{p['payload_type']}]  {p['payload_description'][:60]}")

目标构建体: vB_Eco_TFE_004 (ENG:CONST:3DB53E9B)
✅ payloads: ENG:PAYLOAD:4192FB98, ENG:PAYLOAD:B2DB7B84

=== 所有 DeliveryPayload ===
  ENG:PAYLOAD:AFED61FD  [biofilm_enzyme]  验收：biofilm degrading enzyme
  ENG:PAYLOAD:4192FB98  [biofilm_enzyme]  Displays biofilm-degrading enzyme on phage capsid
  ENG:PAYLOAD:B2DB7B84  [crispr_cas]  Encodes CRISPR-Cas3 to target antibiotic resistance genes


In [6]:
# Cell 6: EngineeringModification
mod1 = create_engineering_modification(
    driver,
    modification_type="replacement",
    modification_summary="Replaced tail fiber gene with chimeric construct to expand host range",
    construct_id=cid["cid"],
    target_component="tail_fiber",
    actor_id="demo",
)
mod2 = create_engineering_modification(
    driver,
    modification_type="insertion",
    modification_summary="Inserted lysin gene under late promoter to enhance lysis",
    construct_id=cid["cid"],
    target_component="lysis_module",
    actor_id="demo",
)
print(f"✅ modifications: {mod1}, {mod2}")

print("\n=== 所有 EngineeringModification ===")
for m in list_modifications(driver):
    print(f"  {m['modification_id']}  [{m['modification_type']}]  {m['modification_summary'][:70]}")

✅ modifications: ENG:MOD:D7B3840C, ENG:MOD:F60AC498

=== 所有 EngineeringModification ===
  ENG:MOD:C2272135  [replacement]  验收：tail fiber replacement
  ENG:MOD:D7B3840C  [replacement]  Replaced tail fiber gene with chimeric construct to expand host range
  ENG:MOD:F60AC498  [insertion]  Inserted lysin gene under late promoter to enhance lysis


In [7]:
# Cell 7: 全量汇总
with driver.session() as s:
    print("=== PRD 对象统计 ===")
    for label, q in [
        ("ClinicalIndication", "MATCH (ci:ClinicalIndication) RETURN count(ci) AS c"),
        ("ProductAsset", "MATCH (a:ProductAsset) RETURN count(a) AS c"),
        ("DecisionOption", "MATCH (o:DecisionOption) RETURN count(o) AS c"),
        ("DeliveryPayload", "MATCH (pl:DeliveryPayload) RETURN count(pl) AS c"),
        ("EngineeringModification", "MATCH (em:EngineeringModification) RETURN count(em) AS c"),
    ]:
        cnt = s.run(q).single()["c"]
        print(f"  {label:30s} {cnt}")

    print("\n=== 关系统计 ===")
    for rel, q in [
        ("ADDRESSES", "MATCH ()-[r:ADDRESSES]->() RETURN count(r) AS c"),
        ("OWNS_ASSET", "MATCH ()-[r:OWNS_ASSET]->() RETURN count(r) AS c"),
        ("OPTION_FOR", "MATCH ()-[r:OPTION_FOR]->() RETURN count(r) AS c"),
        ("CARRIES_PAYLOAD", "MATCH ()-[r:CARRIES_PAYLOAD]->() RETURN count(r) AS c"),
        ("HAS_MODIFICATION", "MATCH ()-[r:HAS_MODIFICATION]->() RETURN count(r) AS c"),
    ]:
        cnt = s.run(q).single()["c"]
        print(f"  {rel:30s} {cnt}")

=== PRD 对象统计 ===
  ClinicalIndication             3
  ProductAsset                   2
  DecisionOption                 3
  DeliveryPayload                3
  EngineeringModification        3

=== 关系统计 ===
  ADDRESSES                      2
  OWNS_ASSET                     2
  OPTION_FOR                     3
  CARRIES_PAYLOAD                3
  HAS_MODIFICATION               3
